# 在 Notebook 中提问 LightRAG

这个 notebook 用来直接调用项目的 `/chat` 接口，不用手写 `curl`。

启动方式：

```bash
python -m agentic_rag.main serve
```

默认 API 地址按 `APP_PORT=8010` 处理，避免和 `.env` 里的 vLLM `8000` 冲突。

In [ ]:
import json
from pathlib import Path
from pprint import pprint

import pandas as pd
import requests

API_BASE = "http://127.0.0.1:8010"
TRIPLES_PATH = Path("../lightrag/standard_triples.json")


def ask(question: str, force_route: str = "lightrag"):
    payload = {
        "question": question,
        "force_route": force_route,
    }
    response = requests.post(f"{API_BASE}/chat", json=payload, timeout=120)
    response.raise_for_status()
    data = response.json()
    answer = data.get("answer", "")
    print("路由:", data.get("route"))
    print("原因:", data.get("reason"))
    print("\n回答:\n")
    print(answer if answer else "<空回答>")
    return data


def load_triples(path: Path = TRIPLES_PATH):
    if not path.exists():
        raise FileNotFoundError(f"未找到标准三元组文件: {path.resolve()}")
    return json.loads(path.read_text(encoding="utf-8"))


def triples_df(path: Path = TRIPLES_PATH):
    triples = load_triples(path)
    return pd.DataFrame(triples)


def search_triples(keyword: str, path: Path = TRIPLES_PATH, limit: int = 20):
    df = triples_df(path)
    lowered = keyword.lower()
    mask = (
        df["subject"].str.lower().str.contains(lowered, na=False)
        | df["predicate"].str.lower().str.contains(lowered, na=False)
        | df["object"].str.lower().str.contains(lowered, na=False)
        | df["evidence"].str.lower().str.contains(lowered, na=False)
    )
    return df.loc[mask].head(limit)


In [ ]:
# 先测一下服务是否正常
requests.get(f"{API_BASE}/health", timeout=30).json()

In [ ]:
ask("这些PDF中有哪些核心实体及其关系？")

In [ ]:
ask("ASME B16.5-2013 中，Ring-Joint Facings 相关的核心实体有哪些，它们分别对应哪些表格、尺寸信息和压力等级？")

In [ ]:
ask("文档里哪些材料、温度阈值和相变或材料风险之间存在关系？例如 425°C、875°F、Graphite、Carbon-Molybdenum Steel 是怎么关联的？")

## 查看标准三元组

重新执行入库后，项目会在 `lightrag/standard_triples.json` 中写入标准谓语三元组。下面几个单元格用来检查导出结果。

In [ ]:
# 查看文件是否存在，以及总共有多少条三元组
triples = load_triples()
print("标准三元组文件:", TRIPLES_PATH.resolve())
print("三元组总数:", len(triples))

In [ ]:
# 随机看前几条
pprint(triples[:5])

In [ ]:
# 以表格形式查看，便于筛选
df = triples_df()
df.head(10)

In [ ]:
# 按关键词筛选，例如查询 ASME B16.5-2013 的相关三元组
search_triples("ASME B16.5-2013")

In [ ]:
# 查询温度和材料相关的标准三元组
search_triples("Graphite")